In [1]:
import pandas as pd
import os
from geopy.geocoders import Nominatim
import pandas as pd
import time
import numpy as np

In [2]:
# Caminho que você está tentando usar
caminho = "/home/felipe/Projeto/Portfolio/Portfolio2/Regression_PriceHouse/src//data/raw/imoveis_sao_paulo_completo.csv"

# Verifique se o arquivo existe
print("O arquivo existe?", os.path.exists(caminho))
print("É um arquivo?", os.path.isfile(caminho) if os.path.exists(caminho) else "Arquivo não encontrado")

O arquivo existe? True
É um arquivo? True


In [3]:
df = pd.read_csv(caminho)

df

,preco,condominio,iptu,rua,endereco,quartos,banheiros,area_m2,link,cidade,estado,data_coleta
0,R$ 14.000,R$ 2.400,R$ 835,NaN,"Apartamento para alugar com 190 m², 3 quartos,...",3.0,3,190.0,https://www.zapimoveis.com.br/imovel/aluguel-a...,sao-paulo,sp,2026-03-13 15:58:49
1,R$ 21.000,R$ 3.800,R$ 2.200,Avenida Roque Petroni Júnior,"Apartamento para alugar com 197 m², 3 quartos,...",3.0,5,197.0,https://www.zapimoveis.com.br/imovel/venda-apa...,sao-paulo,sp,2026-03-13 15:58:49
2,R$ 7.500,NaN,NaN,NaN,"Sobrado para alugar com 400 m², 5 quartos, 6 b...",5.0,6,400.0,https://www.zapimoveis.com.br/imovel/aluguel-s...,sao-paulo,sp,2026-03-13 15:58:49
3,R$ 1.400,NaN,R$ 100,Avenida Imirim,"Apartamento para alugar com 45 m², 2 quartos, ...",2.0,2,45.0,https://www.zapimoveis.com.br/imovel/aluguel-a...,sao-paulo,sp,2026-03-13 15:58:49
4,R$ 4.500,R$ 1.050,R$ 350,Alameda dos Aicás,"Apartamento para alugar com 62 m², 2 quartos, ...",2.0,1,62.0,https://www.zapimoveis.com.br/imovel/aluguel-a...,sao-paulo,sp,2026-03-13 15:58:49
...,...,...,...,...,...,...,...,...,...,...,...,...
6633,R$ 2.550,R$ 1.900,R$ 300,Rua Castelhano,"Apartamento para alugar com 50 m², 1 quarto, 1...",1.0,1,50.0,https://www.zapimoveis.com.br/imovel/aluguel-a...,sao-paulo,sp,2026-03-13 21:51:40
6634,R$ 2.000,R$ 2.600,R$ 500,Rua Vieira de Morais,"Flat para alugar com 47 m², 1 quarto, 1 banhei...",1.0,1,47.0,https://www.zapimoveis.com.br/imovel/aluguel-f...,sao-paulo,sp,2026-03-13 21:51:40
6635,R$ 5.990,R$ 878,R$ 278,Rua Eugênio de Medeiros,"Apartamento para alugar com 34 m², 1 quarto, 1...",1.0,1,34.0,https://www.zapimoveis.com.br/imovel/aluguel-a...,sao-paulo,sp,2026-03-13 21:51:40
6636,R$ 5.100,R$ 700,R$ 300,Rua Leopoldo Couto Magalhães Júnior,"Apartamento para alugar com 37 m², 1 quarto, 1...",1.0,1,37.0,https://www.zapimoveis.com.br/imovel/aluguel-a...,sao-paulo,sp,2026-03-13 21:51:40


In [4]:
df["vagas_garagem"] = (
    df["endereco"]
    .str.extract(r"(\d+)\s+vagas?", expand=False)
    .astype(float)
)

df["bairro"] = (
    df["endereco"]
    .str.extract(r"em\s*([^,]+)", expand=False)
    .str.replace(r"^Imóvel\s+", "", regex=True)
    .str.strip()
)

df["tipo_imovel"] = (
    df["endereco"]
    .str.extract(r"^(.*?)\s+para alugar", expand=False)
    .str.strip()
)

df["preco"] = (
    df["preco"]
    .str.replace("R$", "", regex=False)
    .str.replace(".", "", regex=False)
    .str.replace(",", ".", regex=False)
    .str.strip()
    .astype(float)
)

df["condominio"] = (
    df["condominio"]
    .str.replace("R$", "", regex=False)
    .str.replace(".", "", regex=False)
    .str.replace(",", ".", regex=False)
    .str.strip()
    .astype(float)
)

df["iptu"] = (
    df["iptu"]
    .str.replace("R$", "", regex=False)
    .str.replace(".", "", regex=False)
    .str.replace(",", ".", regex=False)
    .str.strip()
    .astype(float)
)


df

,preco,condominio,iptu,rua,endereco,quartos,banheiros,area_m2,link,cidade,estado,data_coleta,vagas_garagem,bairro,tipo_imovel
0,14000.0,2400.0,835.0,NaN,"Apartamento para alugar com 190 m², 3 quartos,...",3.0,3,190.0,https://www.zapimoveis.com.br/imovel/aluguel-a...,sao-paulo,sp,2026-03-13 15:58:49,2.0,Itaim Bibi,Apartamento
1,21000.0,3800.0,2200.0,Avenida Roque Petroni Júnior,"Apartamento para alugar com 197 m², 3 quartos,...",3.0,5,197.0,https://www.zapimoveis.com.br/imovel/venda-apa...,sao-paulo,sp,2026-03-13 15:58:49,3.0,Jardim das Acácias,Apartamento
2,7500.0,NaN,NaN,NaN,"Sobrado para alugar com 400 m², 5 quartos, 6 b...",5.0,6,400.0,https://www.zapimoveis.com.br/imovel/aluguel-s...,sao-paulo,sp,2026-03-13 15:58:49,1.0,Vila Cavaton,Sobrado
3,1400.0,NaN,100.0,Avenida Imirim,"Apartamento para alugar com 45 m², 2 quartos, ...",2.0,2,45.0,https://www.zapimoveis.com.br/imovel/aluguel-a...,sao-paulo,sp,2026-03-13 15:58:49,NaN,Imirim,Apartamento
4,4500.0,1050.0,350.0,Alameda dos Aicás,"Apartamento para alugar com 62 m², 2 quartos, ...",2.0,1,62.0,https://www.zapimoveis.com.br/imovel/aluguel-a...,sao-paulo,sp,2026-03-13 15:58:49,1.0,Indianópolis,Apartamento
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6633,2550.0,1900.0,300.0,Rua Castelhano,"Apartamento para alugar com 50 m², 1 quarto, 1...",1.0,1,50.0,https://www.zapimoveis.com.br/imovel/aluguel-a...,sao-paulo,sp,2026-03-13 21:51:40,1.0,Vila Andrade,Apartamento
6634,2000.0,2600.0,500.0,Rua Vieira de Morais,"Flat para alugar com 47 m², 1 quarto, 1 banhei...",1.0,1,47.0,https://www.zapimoveis.com.br/imovel/aluguel-f...,sao-paulo,sp,2026-03-13 21:51:40,1.0,Campo Belo,Flat
6635,5990.0,878.0,278.0,Rua Eugênio de Medeiros,"Apartamento para alugar com 34 m², 1 quarto, 1...",1.0,1,34.0,https://www.zapimoveis.com.br/imovel/aluguel-a...,sao-paulo,sp,2026-03-13 21:51:40,1.0,Pinheiros,Apartamento
6636,5100.0,700.0,300.0,Rua Leopoldo Couto Magalhães Júnior,"Apartamento para alugar com 37 m², 1 quarto, 1...",1.0,1,37.0,https://www.zapimoveis.com.br/imovel/aluguel-a...,sao-paulo,sp,2026-03-13 21:51:40,NaN,Itaim Bibi,Apartamento


In [5]:
df["preco_m2"] = df["preco"] / df["area_m2"]

df

,preco,condominio,iptu,rua,endereco,quartos,banheiros,area_m2,link,cidade,estado,data_coleta,vagas_garagem,bairro,tipo_imovel,preco_m2
0,14000.0,2400.0,835.0,NaN,"Apartamento para alugar com 190 m², 3 quartos,...",3.0,3,190.0,https://www.zapimoveis.com.br/imovel/aluguel-a...,sao-paulo,sp,2026-03-13 15:58:49,2.0,Itaim Bibi,Apartamento,73.684211
1,21000.0,3800.0,2200.0,Avenida Roque Petroni Júnior,"Apartamento para alugar com 197 m², 3 quartos,...",3.0,5,197.0,https://www.zapimoveis.com.br/imovel/venda-apa...,sao-paulo,sp,2026-03-13 15:58:49,3.0,Jardim das Acácias,Apartamento,106.598985
2,7500.0,NaN,NaN,NaN,"Sobrado para alugar com 400 m², 5 quartos, 6 b...",5.0,6,400.0,https://www.zapimoveis.com.br/imovel/aluguel-s...,sao-paulo,sp,2026-03-13 15:58:49,1.0,Vila Cavaton,Sobrado,18.750000
3,1400.0,NaN,100.0,Avenida Imirim,"Apartamento para alugar com 45 m², 2 quartos, ...",2.0,2,45.0,https://www.zapimoveis.com.br/imovel/aluguel-a...,sao-paulo,sp,2026-03-13 15:58:49,NaN,Imirim,Apartamento,31.111111
4,4500.0,1050.0,350.0,Alameda dos Aicás,"Apartamento para alugar com 62 m², 2 quartos, ...",2.0,1,62.0,https://www.zapimoveis.com.br/imovel/aluguel-a...,sao-paulo,sp,2026-03-13 15:58:49,1.0,Indianópolis,Apartamento,72.580645
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6633,2550.0,1900.0,300.0,Rua Castelhano,"Apartamento para alugar com 50 m², 1 quarto, 1...",1.0,1,50.0,https://www.zapimoveis.com.br/imovel/aluguel-a...,sao-paulo,sp,2026-03-13 21:51:40,1.0,Vila Andrade,Apartamento,51.000000
6634,2000.0,2600.0,500.0,Rua Vieira de Morais,"Flat para alugar com 47 m², 1 quarto, 1 banhei...",1.0,1,47.0,https://www.zapimoveis.com.br/imovel/aluguel-f...,sao-paulo,sp,2026-03-13 21:51:40,1.0,Campo Belo,Flat,42.553191
6635,5990.0,878.0,278.0,Rua Eugênio de Medeiros,"Apartamento para alugar com 34 m², 1 quarto, 1...",1.0,1,34.0,https://www.zapimoveis.com.br/imovel/aluguel-a...,sao-paulo,sp,2026-03-13 21:51:40,1.0,Pinheiros,Apartamento,176.176471
6636,5100.0,700.0,300.0,Rua Leopoldo Couto Magalhães Júnior,"Apartamento para alugar com 37 m², 1 quarto, 1...",1.0,1,37.0,https://www.zapimoveis.com.br/imovel/aluguel-a...,sao-paulo,sp,2026-03-13 21:51:40,NaN,Itaim Bibi,Apartamento,137.837838


In [6]:
df["endereco_geo"] = (
    df["rua"].str.strip() + ", " +
    df["bairro"].str.strip() + ", São Paulo, SP, Brasil"
)


In [7]:
#Verificar os endereços unicos
enderecos_unicos = df["endereco_geo"].dropna().unique()
len(enderecos_unicos)

1319

In [ ]:
%%time

geolocator = Nominatim(user_agent="imoveis-ml")

#def geocode_endereco(bairro):
#    try:
#        location = geolocator.geocode(
#            f"{bairro}, Ponta Grossa, PR, Brasil",
#            timeout=10
#        )
#        time.sleep(1)  # obrigatório para Nominatim
#        if location:
#            return location.latitude, location.longitude
#    except:
#        return None, None

#df[["lat", "lon"]] = df["bairro"].apply(
#    lambda x: pd.Series(geocode_endereco(x))
#)

cache = {}

for i, endereco in enumerate(enderecos_unicos, start=1):
    if endereco in cache:
        continue

    try:
        location = geolocator.geocode(endereco, timeout=10)
        time.sleep(1)  # obrigatório (Nominatim)

        if location:
            cache[endereco] = (location.latitude, location.longitude)
        else:
            cache[endereco] = (None, None)

    except Exception as e:
        cache[endereco] = (None, None)

    if i % 50 == 0:
        print(f"{i}/{len(enderecos_unicos)} endereços processados")


In [ ]:
#salvar o cache
cache_df = (
    pd.DataFrame.from_dict(
        cache,
        orient="index",
        columns=["lat", "lon"]
    )
    .reset_index()
    .rename(columns={"index": "endereco_geo"})
)

cache_df.to_csv("cache_geocoding.csv", index=False)


In [8]:
cache_df = pd.read_csv("cache_geocoding.csv")
cache_df

,endereco_geo,lat,lon
0,"Avenida Roque Petroni Júnior, Jardim das Acáci...",NaN,NaN
1,"Avenida Imirim, Imirim, São Paulo, SP, Brasil",-23.496380,-46.637037
2,"Alameda dos Aicás, Indianópolis, São Paulo, SP...",-23.610209,-46.659335
3,"Rua Doutor Bento Teobaldo Ferraz, Várzea da Ba...",NaN,NaN
4,"Rua Intendência, Brás, São Paulo, SP, Brasil",NaN,NaN
...,...,...,...
1314,"Rua Florianópolis, Vila Bertioga, São Paulo, S...",NaN,NaN
1315,"Rua Tijuco Preto, Tatuapé, São Paulo, SP, Brasil",-23.543990,-46.571665
1316,"Estrada Velha da Penha, Tatuapé, São Paulo, SP...",-23.528785,-46.555738
1317,"Avenida Parada Pinto, Vila Nova Cachoeirinha, ...",NaN,NaN


In [9]:
#unir o conjunto de dados com o cache
df = df.merge(
    cache_df,
    on="endereco_geo",
    how="left"
)

df

,preco,condominio,iptu,rua,endereco,quartos,banheiros,area_m2,link,cidade,estado,data_coleta,vagas_garagem,bairro,tipo_imovel,preco_m2,endereco_geo,lat,lon
0,14000.0,2400.0,835.0,NaN,"Apartamento para alugar com 190 m², 3 quartos,...",3.0,3,190.0,https://www.zapimoveis.com.br/imovel/aluguel-a...,sao-paulo,sp,2026-03-13 15:58:49,2.0,Itaim Bibi,Apartamento,73.684211,NaN,NaN,NaN
1,21000.0,3800.0,2200.0,Avenida Roque Petroni Júnior,"Apartamento para alugar com 197 m², 3 quartos,...",3.0,5,197.0,https://www.zapimoveis.com.br/imovel/venda-apa...,sao-paulo,sp,2026-03-13 15:58:49,3.0,Jardim das Acácias,Apartamento,106.598985,"Avenida Roque Petroni Júnior, Jardim das Acáci...",NaN,NaN
2,7500.0,NaN,NaN,NaN,"Sobrado para alugar com 400 m², 5 quartos, 6 b...",5.0,6,400.0,https://www.zapimoveis.com.br/imovel/aluguel-s...,sao-paulo,sp,2026-03-13 15:58:49,1.0,Vila Cavaton,Sobrado,18.750000,NaN,NaN,NaN
3,1400.0,NaN,100.0,Avenida Imirim,"Apartamento para alugar com 45 m², 2 quartos, ...",2.0,2,45.0,https://www.zapimoveis.com.br/imovel/aluguel-a...,sao-paulo,sp,2026-03-13 15:58:49,NaN,Imirim,Apartamento,31.111111,"Avenida Imirim, Imirim, São Paulo, SP, Brasil",-23.496380,-46.637037
4,4500.0,1050.0,350.0,Alameda dos Aicás,"Apartamento para alugar com 62 m², 2 quartos, ...",2.0,1,62.0,https://www.zapimoveis.com.br/imovel/aluguel-a...,sao-paulo,sp,2026-03-13 15:58:49,1.0,Indianópolis,Apartamento,72.580645,"Alameda dos Aicás, Indianópolis, São Paulo, SP...",-23.610209,-46.659335
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6633,2550.0,1900.0,300.0,Rua Castelhano,"Apartamento para alugar com 50 m², 1 quarto, 1...",1.0,1,50.0,https://www.zapimoveis.com.br/imovel/aluguel-a...,sao-paulo,sp,2026-03-13 21:51:40,1.0,Vila Andrade,Apartamento,51.000000,"Rua Castelhano, Vila Andrade, São Paulo, SP, B...",-23.631448,-46.738618
6634,2000.0,2600.0,500.0,Rua Vieira de Morais,"Flat para alugar com 47 m², 1 quarto, 1 banhei...",1.0,1,47.0,https://www.zapimoveis.com.br/imovel/aluguel-f...,sao-paulo,sp,2026-03-13 21:51:40,1.0,Campo Belo,Flat,42.553191,"Rua Vieira de Morais, Campo Belo, São Paulo, S...",-23.620967,-46.673241
6635,5990.0,878.0,278.0,Rua Eugênio de Medeiros,"Apartamento para alugar com 34 m², 1 quarto, 1...",1.0,1,34.0,https://www.zapimoveis.com.br/imovel/aluguel-a...,sao-paulo,sp,2026-03-13 21:51:40,1.0,Pinheiros,Apartamento,176.176471,"Rua Eugênio de Medeiros, Pinheiros, São Paulo,...",-23.567839,-46.700361
6636,5100.0,700.0,300.0,Rua Leopoldo Couto Magalhães Júnior,"Apartamento para alugar com 37 m², 1 quarto, 1...",1.0,1,37.0,https://www.zapimoveis.com.br/imovel/aluguel-a...,sao-paulo,sp,2026-03-13 21:51:40,NaN,Itaim Bibi,Apartamento,137.837838,"Rua Leopoldo Couto Magalhães Júnior, Itaim Bib...",-23.589205,-46.683923


## 1️⃣ Identificar quem falhou

In [10]:
mask_falha = df["lat"].isna() | df["lon"].isna()

df_falha = df.loc[mask_falha, ["rua", "bairro"]].drop_duplicates()
df_falha.shape


(593, 2)

## 2️⃣ Tentar novamente: Rua + Cidade (sem bairro)

In [11]:
df_falha["endereco_alt"] = (
    df_falha["rua"].str.strip() +
    ", São Paulo, SP, Brasil"
)

cache_alt = {}

for endereco in df_falha["endereco_alt"].unique():
    try:
        location = geolocator.geocode(endereco, timeout=3)
        time.sleep(1)

        if location:
            cache_alt[endereco] = (location.latitude, location.longitude)
        else:
            cache_alt[endereco] = (None, None)

    except:
        cache_alt[endereco] = (None, None)


## 3️⃣ Aplicar o resultado no dataframe principal

In [12]:
cache_alt_df = (
    pd.DataFrame.from_dict(
        cache_alt,
        orient="index",
        columns=["lat_alt", "lon_alt"]
    )
    .reset_index()
    .rename(columns={"index": "endereco_alt"})
)

df = df.merge(
    cache_alt_df,
    left_on=(
        df["rua"].str.strip() + ", Ponta Grossa, PR, Brasil"
    ),
    right_on="endereco_alt",
    how="left"
)

df["lat"] = df["lat"].fillna(df["lat_alt"])
df["lon"] = df["lon"].fillna(df["lon_alt"])

df.drop(columns=["lat_alt", "lon_alt", "endereco_alt"], inplace=True)


/tmp/ipykernel_7709/3088140622.py:20: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["lat"] = df["lat"].fillna(df["lat_alt"])
/tmp/ipykernel_7709/3088140622.py:21: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["lon"] = df["lon"].fillna(df["lon_alt"])


In [13]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6638 entries, 0 to 6637
Data columns (total 19 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   preco          6631 non-null   float64
 1   condominio     6102 non-null   float64
 2   iptu           6143 non-null   float64
 3   rua            6529 non-null   object 
 4   endereco       6638 non-null   object 
 5   quartos        6634 non-null   float64
 6   banheiros      6638 non-null   int64  
 7   area_m2        6638 non-null   float64
 8   link           6638 non-null   object 
 9   cidade         6638 non-null   object 
 10  estado         6638 non-null   object 
 11  data_coleta    6638 non-null   object 
 12  vagas_garagem  5726 non-null   float64
 13  bairro         6638 non-null   object 
 14  tipo_imovel    6638 non-null   object 
 15  preco_m2       6631 non-null   float64
 16  endereco_geo   6529 non-null   object 
 17  lat            4360 non-null   float64
 18  lon     

## 4️⃣ Fallback final: Bairro + Cidade (último recurso)

In [14]:
mask_final = df["lat"].isna()

df.loc[mask_final, "endereco_geo"] = (
    df.loc[mask_final, "bairro"] +
    ", São Paulo, SP, Brasil"
)


In [15]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6638 entries, 0 to 6637
Data columns (total 19 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   preco          6631 non-null   float64
 1   condominio     6102 non-null   float64
 2   iptu           6143 non-null   float64
 3   rua            6529 non-null   object 
 4   endereco       6638 non-null   object 
 5   quartos        6634 non-null   float64
 6   banheiros      6638 non-null   int64  
 7   area_m2        6638 non-null   float64
 8   link           6638 non-null   object 
 9   cidade         6638 non-null   object 
 10  estado         6638 non-null   object 
 11  data_coleta    6638 non-null   object 
 12  vagas_garagem  5726 non-null   float64
 13  bairro         6638 non-null   object 
 14  tipo_imovel    6638 non-null   object 
 15  preco_m2       6631 non-null   float64
 16  endereco_geo   6638 non-null   object 
 17  lat            4360 non-null   float64
 18  lon     

## 📊 Métrica de qualidade (recomendo salvar)

In [16]:
df["nivel_geocoding"] = np.select(
    [
        df["lat"].notna() & df["rua"].notna() & df["bairro"].notna(),
        df["lat"].notna() & df["rua"].notna(),
        df["lat"].notna() & df["bairro"].notna()
    ],
    [
        "rua_bairro",
        "rua",
        "bairro"
    ],
    default="falhou"
)

df

,preco,condominio,iptu,rua,endereco,quartos,banheiros,area_m2,link,cidade,estado,data_coleta,vagas_garagem,bairro,tipo_imovel,preco_m2,endereco_geo,lat,lon,nivel_geocoding
0,14000.0,2400.0,835.0,NaN,"Apartamento para alugar com 190 m², 3 quartos,...",3.0,3,190.0,https://www.zapimoveis.com.br/imovel/aluguel-a...,sao-paulo,sp,2026-03-13 15:58:49,2.0,Itaim Bibi,Apartamento,73.684211,"Itaim Bibi, São Paulo, SP, Brasil",NaN,NaN,falhou
1,21000.0,3800.0,2200.0,Avenida Roque Petroni Júnior,"Apartamento para alugar com 197 m², 3 quartos,...",3.0,5,197.0,https://www.zapimoveis.com.br/imovel/venda-apa...,sao-paulo,sp,2026-03-13 15:58:49,3.0,Jardim das Acácias,Apartamento,106.598985,"Jardim das Acácias, São Paulo, SP, Brasil",NaN,NaN,falhou
2,7500.0,NaN,NaN,NaN,"Sobrado para alugar com 400 m², 5 quartos, 6 b...",5.0,6,400.0,https://www.zapimoveis.com.br/imovel/aluguel-s...,sao-paulo,sp,2026-03-13 15:58:49,1.0,Vila Cavaton,Sobrado,18.750000,"Vila Cavaton, São Paulo, SP, Brasil",NaN,NaN,falhou
3,1400.0,NaN,100.0,Avenida Imirim,"Apartamento para alugar com 45 m², 2 quartos, ...",2.0,2,45.0,https://www.zapimoveis.com.br/imovel/aluguel-a...,sao-paulo,sp,2026-03-13 15:58:49,NaN,Imirim,Apartamento,31.111111,"Avenida Imirim, Imirim, São Paulo, SP, Brasil",-23.496380,-46.637037,rua_bairro
4,4500.0,1050.0,350.0,Alameda dos Aicás,"Apartamento para alugar com 62 m², 2 quartos, ...",2.0,1,62.0,https://www.zapimoveis.com.br/imovel/aluguel-a...,sao-paulo,sp,2026-03-13 15:58:49,1.0,Indianópolis,Apartamento,72.580645,"Alameda dos Aicás, Indianópolis, São Paulo, SP...",-23.610209,-46.659335,rua_bairro
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6633,2550.0,1900.0,300.0,Rua Castelhano,"Apartamento para alugar com 50 m², 1 quarto, 1...",1.0,1,50.0,https://www.zapimoveis.com.br/imovel/aluguel-a...,sao-paulo,sp,2026-03-13 21:51:40,1.0,Vila Andrade,Apartamento,51.000000,"Rua Castelhano, Vila Andrade, São Paulo, SP, B...",-23.631448,-46.738618,rua_bairro
6634,2000.0,2600.0,500.0,Rua Vieira de Morais,"Flat para alugar com 47 m², 1 quarto, 1 banhei...",1.0,1,47.0,https://www.zapimoveis.com.br/imovel/aluguel-f...,sao-paulo,sp,2026-03-13 21:51:40,1.0,Campo Belo,Flat,42.553191,"Rua Vieira de Morais, Campo Belo, São Paulo, S...",-23.620967,-46.673241,rua_bairro
6635,5990.0,878.0,278.0,Rua Eugênio de Medeiros,"Apartamento para alugar com 34 m², 1 quarto, 1...",1.0,1,34.0,https://www.zapimoveis.com.br/imovel/aluguel-a...,sao-paulo,sp,2026-03-13 21:51:40,1.0,Pinheiros,Apartamento,176.176471,"Rua Eugênio de Medeiros, Pinheiros, São Paulo,...",-23.567839,-46.700361,rua_bairro
6636,5100.0,700.0,300.0,Rua Leopoldo Couto Magalhães Júnior,"Apartamento para alugar com 37 m², 1 quarto, 1...",1.0,1,37.0,https://www.zapimoveis.com.br/imovel/aluguel-a...,sao-paulo,sp,2026-03-13 21:51:40,NaN,Itaim Bibi,Apartamento,137.837838,"Rua Leopoldo Couto Magalhães Júnior, Itaim Bib...",-23.589205,-46.683923,rua_bairro


In [17]:
df["nivel_geocoding"].value_counts(normalize=True) * 100

nivel_geocoding
rua_bairro    65.682434
falhou        34.317566
Name: proportion, dtype: float64

In [18]:
# Caminho que você está tentando usar
caminho2 = "/home/felipe/Projeto/Portfolio/Portfolio2/Regression_PriceHouse/data/pre/sp_preprocessed.csv"

df.to_csv(caminho2, index=False)

## Importar e salvar os conjuntos de dados referente a feature engineering. 

In [19]:
import osmnx as ox

cidade = "São Paulo, São Paulo, Brasil"

#-----------------------------------------------------
tags_hospitais = {"amenity": "hospital"}

hospitais = ox.features_from_place(cidade, tags_hospitais)
#----------------------------------------------------------

mercados = ox.features_from_place(
    cidade,
    {"shop": ["supermarket", "convenience"]}
)

#---------------------------------------------------------
farmacias = ox.features_from_place(
    cidade,
    {"amenity": "pharmacy"}
)

parques = ox.features_from_place(
    cidade,
    tags={
        "leisure": ["park", "garden"],
        "landuse": "recreation_ground"
    }
)

# -----------------------------------------------------
# Estações de metrô
metro_stations = ox.features_from_place(
    cidade,
    tags={
        "railway": "station",
        "station": "subway"
    }
)

# -----------------------------------------------------
# Entradas de metrô (opcional, mas aumenta cobertura)
metro_entrances = ox.features_from_place(
    cidade,
    tags={"railway": "subway_entrance"}
)

In [20]:
metro = pd.concat([metro_stations, metro_entrances])
metro

geometry  \
element id                                                               
node    60634870                           POINT (-46.62513 -23.51564)   
        176004373                          POINT (-46.63572 -23.55496)   
        605663613                           POINT (-46.69112 -23.5465)   
        1658464892                         POINT (-46.63361 -23.53707)   
        2380750804                           POINT (-46.6523 -23.5978)   
...                                                                ...   
        12917913487                         POINT (-46.64118 -23.5756)   
        12992303449                         POINT (-46.6126 -23.60198)   
        12992303450                        POINT (-46.61241 -23.60226)   
        13608665318                        POINT (-46.73504 -23.59289)   
way     1095188246   POLYGON ((-46.69251 -23.63362, -46.69231 -23.6...   

                                     name public_transport          railway  \
element id                                                                    
node    60634870       Portuguesa - Tietê    stop_position             stop   
        176004373       Japão - Liberdade    stop_position             stop   
        605663613           Vila Madalena          station          station   
        1658464892                    Luz    stop_position             stop   
        2380750804        AACD – Servidor    stop_position             stop   
...                                   ...              ...              ...   
        12917913487  Parada Metro Paraiso              NaN  subway_entrance   
        12992303449                   NaN              NaN  subway_entrance   
        12992303450                   NaN              NaN  subway_entrance   
        13608665318                   NaN              NaN  subway_entrance   
way     1095188246     Estação Borba Gato              NaN  subway_entrance   

                     ref station subway   old_name  \
element id                                           
node    60634870     TTE  subway    yes        NaN   
        176004373    LIB  subway    yes  Liberdade   
        605663613    VMD  subway    yes        NaN   
        1658464892   LUZ  subway    yes        NaN   
        2380750804   NaN  subway    yes        NaN   
...                  ...     ...    ...        ...   
        12917913487  NaN     NaN    NaN        NaN   
        12992303449    A     NaN    NaN        NaN   
        12992303450    B     NaN    NaN        NaN   
        13608665318  NaN     NaN    NaN        NaN   
way     1095188246   NaN     NaN    NaN        NaN   

                                       operator   wikidata  ... type highway  \
element id                                                  ...                
node    60634870                            NaN        NaN  ...  NaN     NaN   
        176004373                           NaN        NaN  ...  NaN     NaN   
        605663613    Metropolitano de São Paulo   Q5244825  ...  NaN     NaN   
        1658464892                          NaN  Q15243160  ...  NaN     NaN   
        2380750804                          NaN   Q5908613  ...  NaN     NaN   
...                                         ...        ...  ...  ...     ...   
        12917913487                     SPTrans        NaN  ...  NaN     NaN   
        12992303449                         NaN        NaN  ...  NaN     NaN   
        12992303450                         NaN        NaN  ...  NaN     NaN   
        13608665318                         NaN        NaN  ...  NaN     NaN   
way     1095188246                          NaN   Q4944343  ...  NaN     NaN   

                    waterway access entrance       description is_in level  \
element id                                                                   
node    60634870         NaN    NaN      NaN               NaN   NaN   NaN   
        176004373        NaN    NaN      NaN               NaN   NaN   NaN   
        6056636

In [21]:
escolas = ox.features_from_place(
    cidade,
    tags={
        "amenity": ["school", "college", "university"]
    }
)
escolas

geometry  \
element id                                                              
node    507071835                         POINT (-46.73411 -23.52802)   
        525522286                         POINT (-46.70408 -23.63375)   
        609954917                         POINT (-46.73654 -23.52544)   
        638425993                         POINT (-46.59965 -23.54121)   
        696175114                         POINT (-46.69814 -23.63585)   
...                                                               ...   
way     1486512187  POLYGON ((-46.50114 -23.59593, -46.501 -23.595...   
        1486698682  POLYGON ((-46.52864 -23.52461, -46.52867 -23.5...   
        1486707690  POLYGON ((-46.56626 -23.52754, -46.56612 -23.5...   
        1486962631  POLYGON ((-46.56422 -23.53526, -46.56417 -23.5...   
        1488713496  POLYGON ((-46.65993 -23.4808, -46.65988 -23.48...   

                       amenity                                     name  \
element id                                                                
node    507071835      college       Universidade Mogi das Cruzes (UMC)   
        525522286       school                                      NaN   
        609954917   university             Universidade Mogi das Cruzes   
        638425993       school            Colégio Presbiteriano do Brás   
        696175114       school                              Coreo Dança   
...                        ...                                      ...   
way     1486512187      school  E.E. Professor Victorio Américo Fontana   
        1486698682      school              E.E Dom João Maria Ogno Osb   
        1486707690     college                                    Senai   
        1486962631      school                E.E Joao Dias da Silveira   
        1488713496      school    Centro Brasileiro de Insino Intregado   

                   addr:housenumber                    addr:street check_date  \
element id                                                                      
node    507071835               NaN                            NaN        NaN   
        525522286               NaN                            NaN        NaN   
        609954917               550  Avenida Imperatriz Leopoldina        NaN   
        638425993               NaN                            NaN        NaN   
        696175114               NaN                            NaN        NaN   
...                             ...                            ...        ...   
way     1486512187              117                 Rua Nepomuceno        NaN   
        1486698682              304              Rua Maria Carlota        NaN   
        1486707690              NaN                            NaN        NaN   
        1486962631               80               Rua Sousa Breves        NaN   
        1488713496             1482    Rua Joaquim Afonso de Souza        NaN   

                   name:en wheelchair wikidata wikipedia  ...  \
element id                                                ...   
node    507071835      NaN        NaN      NaN       NaN  ...   
        525522286      NaN        NaN      NaN       NaN  ...   
        609954917      NaN        NaN      NaN       NaN  ...   
        638425993      NaN        NaN      NaN       NaN  ...   
        696175114      NaN        NaN      NaN       NaN  ...   
...                    ...        ...      ...       ...  ...   
way     1486512187     NaN        NaN      NaN       NaN  ...   
        1486698682     NaN        NaN      NaN       NaN  ...   
        1486707690     NaN        NaN      NaN       NaN  ...   
        1486962631     NaN        NaN      NaN       NaN  ...   
        1488713496     NaN        NaN      NaN       NaN  ...   

                   payment:mastercard payment:pikepass payment:uta  \
element id                                                           
node    507071835                 NaN              NaN         NaN   
        525522286        

In [22]:
escolas["geometry_std"] = escolas.geometry.apply(
    lambda g: g.centroid if g.geom_type != "Point" else g
)

escolas_std = escolas.set_geometry("geometry_std")
escolas_std.geometry.geom_type.value_counts()


Point    1837
Name: count, dtype: int64

In [23]:
import re
import pandas as pd

def classificar_escola(row):
    nome = str(row.get("name", "")).lower()

    padroes_publicos = [
        "municipal",
        "estadual",
        "federal",
        "instituto federal",
        "colégio estadual",
        "escola municipal",
        "cmei", "emei", "emef", "eef", "eeef",
        "universidade federal",
        "universidade estadual",
        "ifpr", "ifsp", "ifsc", "ifrs"
    ]

    for p in padroes_publicos:
        if p in nome:
            return "publica"

    return "privada"

escolas_std["tipo_escola"] = escolas_std.apply(classificar_escola, axis=1)


In [24]:
escolas_std["tipo_escola"].value_counts(normalize=True)

tipo_escola
privada    0.634731
publica    0.365269
Name: proportion, dtype: float64

In [25]:
policia = ox.features_from_place(
    cidade,
    tags={
        "amenity": ["police", "fire_station", "courthouse"]
    }
)
policia

geometry  \
element id                                                              
node    180576749                         POINT (-46.56461 -23.53986)   
        468049990                         POINT (-46.49832 -23.51011)   
        470109183                         POINT (-46.70454 -23.50965)   
        507071859                         POINT (-46.73081 -23.53873)   
        745730429                         POINT (-46.65028 -23.53932)   
...                                                               ...   
way     1470430534  POLYGON ((-46.64872 -23.45658, -46.64866 -23.4...   
        1473347960  POLYGON ((-46.49213 -23.55385, -46.49213 -23.5...   
        1486530148  POLYGON ((-46.5673 -23.53697, -46.56721 -23.53...   
        1488015882  POLYGON ((-46.56799 -23.53172, -46.56782 -23.5...   
        1488015883  POLYGON ((-46.56809 -23.5314, -46.56755 -23.53...   

                         amenity  \
element id                         
node    180576749   fire_station   
        468049990         police   
        470109183         police   
        507071859         police   
        745730429         police   
...                          ...   
way     1470430534        police   
        1473347960        police   
        1486530148        police   
        1488015882    courthouse   
        1488015883    courthouse   

                                                                 name  \
element id                                                              
node    180576749                          Corpo de Bombeiros - 3º GB   
        468049990                               24º Distrito Policial   
        470109183                                     Policia Militar   
        507071859                              91o. Distrito Policial   
        745730429                               Ponto Policia Militar   
...                                                               ...   
way     1470430534                                                NaN   
        1473347960  66º Distrito Policial / 8ª DDM - Delegacia de ...   
        1486530148                         Polícia Militar - 8º BPM/M   
        1488015882                                Forum Regional VIII   
        1488015883                          TJSP Almoxerifado Central   

                                                             operator  \
element id                                                              
node    180576749   Corpo de Bombeiros Militar do Estado de São Paulo   
        468049990                Polícia Civil do Estado de São Paulo   
        470109183              Polícia Militar do Estado de São Paulo   
        507071859                Polícia Civil do Estado de São Paulo   
        745730429                                                 NaN   
...                                                               ...   
way     1470430534             Polícia Militar do Estado de São Paulo   
        1473347960               Polícia Civil do Estado de São Paulo   
        1486530148             Polícia Militar do Estado de São Paulo   
        1488015882         Tribunal de Justiça do Estado de São Paulo   
        1488015883                                                NaN   

                   operator:wikidata   landuse military operator:short  \
element id                                                               
node    180576749          Q10260738       NaN      NaN            NaN   
        468049990          Q10350993       NaN      NaN            NaN   
        470109183          Q10351025  military   police          PMESP   
        507071859          Q10350993       NaN      NaN            NaN   
        745730429                NaN       NaN      NaN            NaN   
...                              ...       ...      ...            ...   
way     1470430534         Q10351025       NaN      NaN          PMESP   
        1473347960         Q10350993       NaN      NaN            NaN   
  

In [26]:
escolas_std.to_csv(
    "/home/felipe/Projeto/Portfolio/Portfolio2/"
    "Regression_PriceHouse/data/pre/escolas_sp.csv",
    index=False
)

hospitais.to_csv(
    "/home/felipe/Projeto/Portfolio/Portfolio2/"
    "Regression_PriceHouse/data/pre/hospitais_sp.csv",
    index=False
)

parques.to_csv(
    "/home/felipe/Projeto/Portfolio/Portfolio2/"
    "Regression_PriceHouse/data/pre/parques_sp.csv",
    index=False
)

mercados.to_csv(
    "/home/felipe/Projeto/Portfolio/Portfolio2/"
    "Regression_PriceHouse/data/pre/mercados_sp.csv",
    index=False
)

farmacias.to_csv(
    "/home/felipe/Projeto/Portfolio/Portfolio2/"
    "Regression_PriceHouse/data/pre/farmacia_sp.csv",
    index=False
)

policia.to_csv(
    "/home/felipe/Projeto/Portfolio/Portfolio2/"
    "Regression_PriceHouse/data/pre/policia_sp.csv",
    index=False
)

In [27]:
metro.to_csv(
    "/home/felipe/Projeto/Portfolio/Portfolio2/"
    "Regression_PriceHouse/data/pre/metro_sp.csv",
    index=False
)